# 03 — Modeling

Loads `../data/features.csv` (produced by `02_eda_featurization.ipynb`), trains a Random Forest
baseline with ordinary K-fold CV vs. `GroupKFold` (by elemental family and by period) to test
generalization to unseen chemistries, checks for plain overfitting with a held-out 80/20 split, and
runs Lasso feature selection. This notebook only computes results/metrics — all plotting happens in
`04_results_visualization.ipynb`. Outputs: `../data/cv_results.csv`, `../data/predictions.csv`,
`../data/lasso_coefficients.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Imports OK")


In [ ]:
df_features = pd.read_csv("../data/features.csv")

meta_cols = ["material_id", "formula", "crystal_system", "family", "period"]
meta = df_features[meta_cols]
y = df_features["dos_ef"]
X = df_features.drop(columns=meta_cols + ["dos_ef"])

print(f"Loaded {len(X)} materials, {X.shape[1]} features from ../data/features.csv")


---
## Part 8 — Random Forest baseline + GroupKFold cross-validation

We first fit a Random Forest with ordinary (ungrouped) K-fold CV as a baseline — this is the
"interpolation within known families" regime, since a given elemental family will typically appear
in both the train and test fold. We then repeat the evaluation with `GroupKFold` grouped by (a)
elemental family and (b) period, which forces each held-out fold to contain elemental chemistries
the model did not train on. A large gap between the ungrouped and grouped scores is evidence that
the model is mostly interpolating rather than learning transferable composition→DOS(E_F) physics.

In [ ]:
rf = RandomForestRegressor(n_estimators=300, max_features="sqrt", random_state=42, n_jobs=-1)

# --- Baseline: ordinary KFold (no grouping constraint) ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_kfold = -cross_val_score(rf, X, y, cv=kf, scoring="neg_mean_absolute_error", n_jobs=-1)
print(f"Ordinary 5-fold CV MAE: {mae_kfold.mean():.4f} +/- {mae_kfold.std():.4f}")


In [ ]:
def run_group_kfold(X, y, groups, label, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    mae_scores, r2_scores = [], []
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
        rf_fold = RandomForestRegressor(n_estimators=300, max_features="sqrt",
                                         random_state=42, n_jobs=-1)
        rf_fold.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = rf_fold.predict(X.iloc[test_idx])
        mae = mean_absolute_error(y.iloc[test_idx], pred)
        r2 = r2_score(y.iloc[test_idx], pred)
        mae_scores.append(mae)
        r2_scores.append(r2)
        held_out = sorted(pd.unique(groups.iloc[test_idx]))
        print(f"[{label}] Fold {fold}: held-out groups={held_out} "
              f"MAE={mae:.4f} R2={r2:.3f}")
    print(f"\n[{label}] Mean MAE = {np.mean(mae_scores):.4f} +/- {np.std(mae_scores):.4f}, "
          f"Mean R2 = {np.mean(r2_scores):.3f}\n")
    return mae_scores, r2_scores


mae_family, r2_family = run_group_kfold(X, y, meta["family"], "grouped by elemental family")


In [ ]:
mae_period, r2_period = run_group_kfold(X, y, meta["period"], "grouped by period")


**Comparison.** The ordinary K-fold MAE above is the "easy" number — it very likely underestimates
error on genuinely novel chemistry, since related elements tend to land in both train and test folds
by chance. Compare it directly to the family- and period-grouped MAEs: a substantially higher grouped
MAE indicates the model is leaning on family-specific patterns rather than a transferable
composition→DOS(E_F) relationship.

---
## Part 10 — Held-out train/test split

As a second, complementary check, we also fit on a simple random 80/20 train/test split (not grouped
by family — this is deliberately the "easy," interpolation-friendly split) and compare train vs. test
error directly. A large train/test gap here is the classic overfitting signature, independent of the
elemental-generalization question Part 8 addresses.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
meta_train, meta_test = meta.loc[X_train.index], meta.loc[X_test.index]

rf_tt = RandomForestRegressor(n_estimators=300, max_features="sqrt", random_state=42, n_jobs=-1)
rf_tt.fit(X_train, y_train)

pred_train = rf_tt.predict(X_train)
pred_test = rf_tt.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))

print(f"Train RMSE: {rmse_train:.4f}  (n={len(X_train)})")
print(f"Test RMSE:  {rmse_test:.4f}  (n={len(X_test)})")
print(f"Test/train RMSE ratio: {rmse_test / rmse_train:.2f}x")


---
## Part 11 — Lasso feature selection

We standardize the feature matrix and fit `LassoCV` (5-fold CV over a path of `alpha` values) to see
which MAGPIE descriptors and crystal-system indicators actually carry predictive weight for
DOS(E_F), versus which are shrunk to exactly zero.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lasso = LassoCV(cv=5, random_state=42, max_iter=20000, n_jobs=-1).fit(X_scaled, y)

coef = pd.Series(lasso.coef_, index=X.columns)
nonzero = coef[coef != 0].reindex(coef[coef != 0].abs().sort_values(ascending=False).index)

print(f"Selected alpha (LassoCV): {lasso.alpha_:.5f}")
print(f"Lasso retained {len(nonzero)} / {len(coef)} features as nonzero.\n")
print("Top 20 features by |coefficient|:")
nonzero.head(20)


---
## Save results

Three tidy CSVs, one per diagnostic, for `04_results_visualization.ipynb` to plot:
- `cv_results.csv` — mean/std MAE for the three CV schemes (Part 8)
- `predictions.csv` — per-material actual vs. predicted DOS(E_F), tagged by train/test split (Part 10)
- `lasso_coefficients.csv` — standardized Lasso coefficient per feature (Part 11)

In [ ]:
os.makedirs("../data", exist_ok=True)

cv_results = pd.DataFrame({
    "scheme": ["Ordinary K-fold", "GroupKFold (family)", "GroupKFold (period)"],
    "mean_mae": [mae_kfold.mean(), np.mean(mae_family), np.mean(mae_period)],
    "std_mae":  [mae_kfold.std(),  np.std(mae_family),  np.std(mae_period)],
})
cv_results.to_csv("../data/cv_results.csv", index=False)

predictions = pd.concat([
    meta_train.assign(split="train", y_true=y_train.values, y_pred=pred_train),
    meta_test.assign(split="test", y_true=y_test.values, y_pred=pred_test),
], ignore_index=True)
predictions.to_csv("../data/predictions.csv", index=False)

lasso_coefficients = coef.rename("coefficient").rename_axis("feature").reset_index()
lasso_coefficients.to_csv("../data/lasso_coefficients.csv", index=False)

print("Saved ../data/cv_results.csv, ../data/predictions.csv, ../data/lasso_coefficients.csv")
